# AML transaction monitoring: XGBoost vs graph neural network on IBM HI-Medium

Runs the project's v2 pipeline (rules, XGBoost, durability checks) and then the GNN experiment
(directional GraphSAGE) on the same time windows and the same test accounts.

**Before running**
1. *Add data*: the Kaggle dataset **IBM Transactions for Anti Money Laundering (AML)**.
2. *Add your code*: either push the repo to GitHub and set `REPO_URL` below (needs Internet on),
   or upload the project folder as a Kaggle dataset (without `data/`) and leave `REPO_URL` empty.
3. *Settings → Accelerator*: **GPU T4 x2** or **GPU P100**.
4. Recommended: first run with `DATASET = "HI-Small"` (about 30-40 min) to check everything,
   then switch to `"HI-Medium"` (about 2-3 hours). Use *Save Version → Save & Run All* so it runs
   in the background.

Every number this notebook prints comes from the run itself. `resume_bullets_gnn.md` only claims a
GNN gain if the paired bootstrap says it is significant.

In [ ]:
DATASET = "HI-Medium"      # "HI-Small" for a quick check first
REPO_URL = ""              # e.g. "https://github.com/rishysp/aml-transaction-monitoring"
N_WINDOWS = {"HI-Small": 5, "HI-Medium": 30}[DATASET]  # ~1M transactions per window in both cases
N_TRIALS = 10              # XGBoost hyperparameter search trials

In [ ]:
import glob, os, shutil, subprocess, sys
!nvidia-smi --query-gpu=name,memory.total --format=csv
csv = glob.glob(f"/kaggle/input/**/{DATASET}_Trans.csv", recursive=True)
assert csv, f"{DATASET}_Trans.csv not found: add the IBM AML dataset to this notebook"
RAW = csv[0]
print("data:", RAW, f"{os.path.getsize(RAW) / 1e9:.2f} GB")

REPO = "/kaggle/working/repo"
if not os.path.exists(REPO):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
    else:
        found = glob.glob("/kaggle/input/**/scripts/run_pipeline.py", recursive=True)
        assert found, "Code not found: set REPO_URL or add the project folder as a Kaggle dataset"
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), REPO)
os.chdir(REPO)
print("code:", REPO, sorted(os.listdir(REPO)))

In [ ]:
!pip install -q "xgboost>=2.0" networkx pyyaml
import torch, xgboost
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| xgboost", xgboost.__version__)

In [ ]:
import yaml
cfg = yaml.safe_load(open("config.yaml"))
OUT = f"/kaggle/working/reports_{DATASET}"
cfg["data"]["raw_csv"] = RAW
cfg["data"]["processed"] = f"/kaggle/working/{DATASET}.parquet"
cfg["split"]["n_windows"] = N_WINDOWS
cfg["models"]["xgboost"]["device"] = "cuda" if torch.cuda.is_available() else "cpu"
cfg["models"]["search"]["n_trials"] = N_TRIALS
cfg["outputs"] = {"reports_dir": OUT, "figures_dir": f"{OUT}/figures"}
CONFIG = f"/kaggle/working/config_{DATASET}.yaml"
yaml.safe_dump(cfg, open(CONFIG, "w"))
print(open(CONFIG).read())

## 1. Pipeline: windows, rules, XGBoost, durability checks
Builds and caches features for every window, then tunes and evaluates. If the session stops,
rerun this cell with `--reuse-search` added: cached windows and the finished search are reused.

In [ ]:
!python -W ignore scripts/run_pipeline.py --config {CONFIG}

## 2. GNN experiment on the same windows and test accounts

In [ ]:
!python -W ignore scripts/run_gnn.py --config {CONFIG} --device auto

## 3. Results

In [ ]:
import pandas as pd, json
from IPython.display import Image, Markdown, display
display(Markdown("### Pipeline (XGBoost, rules)"))
display(pd.read_csv(f"{OUT}/model_comparison_test.csv").round(4))
display(Markdown("### GNN vs XGBoost (same test accounts)"))
display(pd.read_csv(f"{OUT}/gnn/model_comparison_gnn.csv").round(4))
display(Markdown("### Paired bootstrap: is the GNN better?"))
display(pd.read_csv(f"{OUT}/gnn/paired_differences_gnn.csv").round(4))
display(Image(f"{OUT}/gnn/figures/alert_efficiency_gnn.png"))
display(Markdown(open(f"{OUT}/gnn/resume_bullets_gnn.md").read()))
display(Markdown(open(f"{OUT}/resume_bullets.md").read()))

In [ ]:
# Download this zip from the Output panel; it holds every report, chart and resume draft
shutil.make_archive(f"/kaggle/working/aml_results_{DATASET}", "zip", OUT)
print("saved", f"/kaggle/working/aml_results_{DATASET}.zip")